Copyright (c) 2026, NVIDIA CORPORATION. Licensed under the Apache License, Version 2.0 (the "License") you may not use this file except in compliance with the License. You may obtain a copy of the License at http://www.apache.org/licenses/LICENSE-2.0 Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied. See the License for the specific language governing permissions and limitations under the License.


# Sampling Chemical Space for Drug Discovery using the MolMIM NIM

## What You Will Do

- Generate molecules around a known seed with MolMIM sampling.
- Score generated molecules with QED and Tanimoto similarity.
- Compare unguided sampling with a simple guided optimization loop.


```{note}
This notebook reads the MolMIM endpoint from `MOLMIM_URL`, or from `.openhackathon-nims.env` when the service wrapper generated one. If you have not started the NIM services yet, follow `00_Container_Setup.ipynb` first.
```

In the field of drug discovery, identifying novel and effective compounds is a crucial step in the development of new medicines. Chemical space, the vast expanse of all possible chemical compounds, is a complex landscape that can be difficult to navigate. Molecular generative AI models, such as MolMIM, can be used to sample this space and identify promising compounds.

## Unguided Sampling: Exploring Chemical Space around a Starting Seed Molecule

In this notebook, we will use MolMIM to perform unguided sampling around a starting seed molecule, upadacitinib. Unguided sampling involves generating new molecules without any specific goal or objective in mind. This can be useful for exploring new areas of chemical space and identifying novel compounds that may not have been considered before.

## Guided Optimization: Improving a Seed Molecule's QED Score

In addition to unguided sampling, we will also use MolMIM to perform guided optimization of the same seed molecule, upadacitinib. Guided optimization involves using a specific objective function, such as the Quantitative Estimate of Drug-likeness (QED) score, to guide the generation of new molecules. In this case, we will use the CMA-ES algorithm to optimize the QED score of upadacitinib.

## CMA-ES: A Global Optimization Algorithm

CMA-ES (Covariance Matrix Adaptation Evolution Strategy) is a global optimization algorithm that is well-suited for optimizing complex objective functions such as the QED score. It uses a population of candidate solutions to search for the optimal solution, and adapts the covariance matrix of the search distribution to improve the search process.

## Objectives

In this notebook, we will use MolMIM to:

1. Perform unguided sampling around the seed molecule upadacitinib to explore chemical space.
2. Perform guided optimization of upadacitinib using CMA-ES to improve its QED score.

## Random Sampling Around a Seed Molecule

In the code blocks below, we will step through how to randomly sample molecules in a similar chemical space to a seed molecule, in this case upadacitinib. We begin by importing the necessary libraries.

In [ ]:
import json
import math
import os
import pickle
import sys
import hashlib
from pathlib import Path
from typing import Any, Dict, List, Optional

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from rdkit import Chem, DataStructs, RDLogger
from rdkit.Chem import Descriptors, FilterCatalog
from rdkit.Chem.AllChem import GetMorganFingerprintAsBitVect
from rdkit.Chem.QED import qed
from rdkit.DataStructs import TanimotoSimilarity

RDLogger.DisableLog("rdApp.*")
CWD = Path.cwd().resolve()
PROJECT_ROOT = CWD if (CWD / "scoring").exists() else CWD.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
try:
    from scoring.endpoint_env import load_openhackathon_env
    load_openhackathon_env()
except Exception:
    pass

MOLMIM_URL = os.environ.get("MOLMIM_URL", "http://localhost:8001").rstrip("/")
HEADERS = {"accept": "application/json", "Content-Type": "application/json"}

def _mean_or_nan(values):
    return float(np.mean(values)) if values else np.nan

def molmim_sample(smiles: str, num_molecules: int = 10, scaled_radius: float = 1.0) -> List[str]:
    response = requests.post(
        f"{MOLMIM_URL}/sampling",
        headers=HEADERS,
        json={"sequences": [smiles], "num_molecules": num_molecules, "scaled_radius": scaled_radius},
        timeout=60,
    )
    response.raise_for_status()
    generated = response.json().get("generated", [])
    return generated[0] if generated and isinstance(generated[0], list) else generated

def molmim_hidden(smiles_list: List[str]) -> np.ndarray:
    response = requests.post(
        f"{MOLMIM_URL}/hidden",
        headers=HEADERS,
        json={"sequences": smiles_list},
        timeout=60,
    )
    response.raise_for_status()
    return np.squeeze(np.array(response.json()["hiddens"]))

def molmim_decode(encodings) -> List[str]:
    encodings = np.array(encodings)
    if encodings.ndim == 1:
        encodings = encodings.reshape(1, -1)
    hiddens = np.expand_dims(encodings, axis=1)
    response = requests.post(
        f"{MOLMIM_URL}/decode",
        headers=HEADERS,
        json={"hiddens": hiddens.tolist(), "mask": [[True] for _ in range(hiddens.shape[0])]},
        timeout=60,
    )
    response.raise_for_status()
    return response.json().get("generated", [])

print(f"MolMIM endpoint: {MOLMIM_URL}")


This code block defines a function called `tanimoto_similarity` that calculates the Tanimoto similarity between two molecules. The function takes two parameters: `smiles`, the SMILES string of the molecule to be compared, and `reference`, the SMILES string of the reference molecule. The function first gets the fingerprint parameters, then creates the fingerprint for the reference molecule. It then validates the input molecule by converting its SMILES string to a molecule object and checks if the object is None. If the object is None, it returns 0. Otherwise, it creates the fingerprint for the input molecule and calculates the Tanimoto similarity between the two fingerprints. The function returns the calculated Tanimoto similarity.

In [ ]:
def tanimoto_similarity(smiles, reference: str):
    # Get fingerprint params
    fingerprint_radius_param = 2
    fingerprint_nbits = 2048

    # Handle the reference molecule
    reference_mol = Chem.MolFromSmiles(reference)
    reference_fingerprint = GetMorganFingerprintAsBitVect(
        reference_mol, radius=fingerprint_radius_param, nBits=fingerprint_nbits
    )

    # Validate the other molecule
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return 0

    fingerprint = GetMorganFingerprintAsBitVect(mol, radius=fingerprint_radius_param, nBits=fingerprint_nbits)

    # Calculate and return the Tanimoto similarity
    return TanimotoSimilarity(fingerprint, reference_fingerprint)

This next block defines the molecule of interest, which is the SMILES string representation of the molecule upadacitinib. This molecule will be used as a reference for calculating Tanimoto similarity and QED scores for other molecules in the script.

In [ ]:
# Define the molecule of interest
upadacitinib = 'CCC1CN(CC1C2=CN=C3N2C4=C(NC=C4)N=C3)C(=O)NCC(F)(F)F'

This code block defines the sampling radii and initializes lists and a dictionary to store the results of the sampling process. The `radii` variable is set to a small set of values between 0.3 and 1.0, which will be used as `scaled_radius` values for MolMIM sampling.


In [ ]:
# Define the scaled radii to sample
radii = np.linspace(0.3, 1.0, num=5)

# Initialize lists to store the results
valid_smiles_counts = []
avg_tanimoto_similarities = []
avg_qed_scores = []

# Create a dictionary to store the results
results = {}


This code block processes molecule sampling at different scaled radii. It sends a request to the MolMIM sampling API, extracts generated SMILES strings, filters invalid strings, calculates Tanimoto similarity and QED score for each valid molecule, and stores the results.


In [ ]:
# Loop through each scaled radius
for radius in radii:
    radius_results = {"tanimoto_similarity": [], "qed_score": []}

    generated_molecules = molmim_sample(upadacitinib, num_molecules=10, scaled_radius=float(radius))

    # Filter out invalid SMILES strings
    valid_smiles = [m for m in generated_molecules if Chem.MolFromSmiles(m) is not None]
    radius_results["valid_smiles"] = len(valid_smiles)

    # Calculate Tanimoto similarity and QED score for each valid SMILES string
    for smile in valid_smiles:
        mol = Chem.MolFromSmiles(smile)
        tanimoto = tanimoto_similarity(smile, upadacitinib)
        qed_score = qed(mol)
        radius_results["tanimoto_similarity"].append(tanimoto)
        radius_results["qed_score"].append(qed_score)

    radius_results["tanimoto_similarity"] = _mean_or_nan(radius_results["tanimoto_similarity"])
    radius_results["qed_score"] = _mean_or_nan(radius_results["qed_score"])

    results[float(radius)] = radius_results

# Create a Pandas dataframe from the results
df = pd.DataFrame(results).T.reset_index().rename(columns={"index": "scaled_radius"})
display(df)


The following block creates three plots to visualize the sampling results. The plots show the number of valid SMILES strings, the average Tanimoto similarity, and the average QED score at each scaled radius.


In [ ]:
# Create the plots
plt.figure(figsize=(20, 4))

# Plot the number of valid SMILES strings at each radius
plt.subplot(1, 3, 1)
plt.plot(df["scaled_radius"], df["valid_smiles"])
plt.xlabel("Scaled radius")
plt.ylabel("Number of valid SMILES strings")
plt.title("Valid SMILES at each scaled radius")

# Plot the average Tanimoto similarity at each radius
plt.subplot(1, 3, 2)
plt.plot(df["scaled_radius"], df["tanimoto_similarity"])
plt.xlabel("Scaled radius")
plt.ylabel("Average Tanimoto similarity")
plt.title("Average Tanimoto similarity")

# Plot the average QED score at each radius
plt.subplot(1, 3, 3)
plt.plot(df["scaled_radius"], df["qed_score"])
plt.xlabel("Scaled radius")
plt.ylabel("Average QED score")
plt.title("Average QED score")

plt.tight_layout()
plt.show()


Run the previous plotting cell to visualize the generated molecules in your current environment. The exact set may vary slightly with sampling and service version.

## Guided Molecular Generation with CMA-ES

In contrast to the random sampling of the latent space described above, we can use a black box optimizer, called CMA-ES, to perform guided optimization of a molecule's property through sampling of the latent space. In the blocks below, we use CMA-ES to optimize the QED score of the generated molecules while preseving a level of similary to the seed molecule, upadacitinib.

This first block initializes variables to store the results of the script and sets up the parameters for the experiment. It defines three lists to store the counts of valid SMILES strings, average Tanimoto similarities, and average QED scores. It also creates an empty dictionary to store the results and a list of minimum similarities to be used in the experiment, ranging from 0.1 to 0.7 with 10 evenly spaced values.

In [ ]:
# Initialize lists to store the guided optimization results
valid_smiles_counts = []
avg_tanimoto_similarities = []
avg_qed_scores = []

# Create a dictionary to store the results
results = {}

# Create a list of similarity targets used by the custom score
min_sims = np.linspace(0.1, 0.7, 5)


The following block is the main loop of the script, where it iterates over each minimum similarity value in the `min_sims` list. For each minimum similarity, it generates molecules using the CMA-ES algorithm, filters out invalid SMILES strings, calculates the Tanimoto similarity and QED score for each valid SMILES string, and stores the results. The results are stored in a dictionary called `results`, where the keys are the minimum similarity values and the values are dictionaries containing the number of valid SMILES strings, average Tanimoto similarity, and average QED score. After the loop, the results are converted into a Pandas dataframe for further analysis.

In [ ]:
try:
    import cma
except ImportError as exc:
    raise RuntimeError("Install cma to run the guided optimization section: pip install cma") from exc

seed_hidden = np.ravel(molmim_hidden([upadacitinib]))

for min_sim in min_sims:
    min_sim_results = {"tanimoto_similarity": [], "qed_score": []}
    optimizer = cma.CMAEvolutionStrategy(seed_hidden, 1.0, {"popsize": 10, "verbose": -9})
    collected_smiles = []

    for _ in range(3):
        candidates = optimizer.ask()
        generated_molecules = molmim_decode(candidates)
        scores = []

        for candidate, smile in zip(candidates, generated_molecules):
            mol = Chem.MolFromSmiles(smile)
            if mol is None:
                scores.append(0.0)
                continue

            tanimoto = tanimoto_similarity(smile, upadacitinib)
            qed_score = qed(mol)
            similarity_component = min(1.0, tanimoto / max(float(min_sim), 1e-6))
            score = qed_score + similarity_component
            scores.append(-score)  # CMA-ES minimizes
            collected_smiles.append(smile)
            min_sim_results["tanimoto_similarity"].append(tanimoto)
            min_sim_results["qed_score"].append(qed_score)

        optimizer.tell(candidates, scores)

    valid_smiles = list(dict.fromkeys(s for s in collected_smiles if Chem.MolFromSmiles(s) is not None))
    min_sim_results["valid_smiles"] = len(valid_smiles)
    min_sim_results["tanimoto_similarity"] = _mean_or_nan(min_sim_results["tanimoto_similarity"])
    min_sim_results["qed_score"] = _mean_or_nan(min_sim_results["qed_score"])
    results[float(min_sim)] = min_sim_results

# Create a Pandas dataframe from the results
df = pd.DataFrame(results).T.reset_index().rename(columns={"index": "minimum_similarity"})
display(df)


The following block creates three plots to visualize the results of the script. The first plot shows the number of valid SMILES strings generated at each minimum similarity threshold. The second plot shows the average Tanimoto similarity of the generated molecules at each minimum similarity threshold. The third plot shows the average QED score of the generated molecules at each minimum similarity threshold. The plots are arranged in a single figure with three subplots, and the figure is displayed using the `plt.show()` function.

In [ ]:
# Create the plots
plt.figure(figsize=(20, 4))

# Plot the number of valid SMILES strings at each min_sim
plt.subplot(1, 3, 1)
plt.plot(df['minimum_similarity'], df['valid_smiles'])
plt.xlabel('Minimum similarity')
plt.ylabel('Number of valid SMILES strings')
plt.title('Number of valid SMILES strings at each minimum similarity')

# Plot the average Tanimoto similarity at each radius
plt.subplot(1, 3, 2)
plt.plot(df['minimum_similarity'], df['tanimoto_similarity'])
plt.xlabel('Minimum similarity')
plt.ylabel('Average Tanimoto similarity')
plt.title('Average Tanimoto similarity at each minimum similarity')

# Plot the average QED score at each radius
plt.subplot(1, 3, 3)
plt.plot(df['minimum_similarity'], df['qed_score'])
plt.xlabel('Minimum similarity')
plt.ylabel('Average QED score')
plt.title('Average QED score at each minimum similarity')

plt.tight_layout()
plt.show()

Run the previous plotting cell to visualize the generated molecules in your current environment. The exact set may vary slightly with sampling and service version.

Note that the `min_similarity` parameter is not a hard cutoff, but rather the minimum similarity necessary to receive a full score for the Tanimoto similarity component of the scoring function. Generally the average similarity will increase with increasing `minimum_similarity` thresholds, but the similarity of the generated molecules to the seed is not guaranted to exceed that threshold.